In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
# from tqdm.notebook import tqdm

from parallel_pandas import ParallelPandas
ParallelPandas.initialize(n_cpu=12)

from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer, FunctionTransformer

import pickle
import os
import glob
import json

import warnings 
from IPython.display import display, HTML
warnings.filterwarnings("ignore")
plt.rcParams["patch.force_edgecolor"] = True

import veriwild_clustering as vwc

In [2]:
# Чтение очищенных датасетов
with open(os.path.join('temp', 'efficientnet_cleaned.pickle'), 'rb') as pkl_file:
    efficientnet_cleaned = pickle.load(pkl_file)
    print(f'{"efficientnet_cleaned":<22}', efficientnet_cleaned.shape)

with open(os.path.join('temp', 'osnet_cleaned.pickle'), 'rb') as pkl_file:
    osnet_cleaned = pickle.load(pkl_file)
    print(f'{"osnet_cleaned":<22}', osnet_cleaned.shape)

with open(os.path.join('temp', 'vdc_color_cleaned.pickle'), 'rb') as pkl_file:
    vdc_color_cleaned = pickle.load(pkl_file)
    print(f'{"vdc_color_cleaned":<22}', vdc_color_cleaned.shape)

with open(os.path.join('temp', 'vdc_type_cleaned.pickle'), 'rb') as pkl_file:
    vdc_type_cleaned = pickle.load(pkl_file)
    print(f'{"vdc_type_cleaned":<22}', vdc_type_cleaned.shape)

efficientnet_cleaned   (416314, 2560)
osnet_cleaned          (416314, 511)
vdc_color_cleaned      (416314, 124)
vdc_type_cleaned       (416314, 396)


# 2. Подбор параметров пайплайна

Последовательность действий: 
1. чтение исходных данных
2. предварительное преобразование 
3. стандартизация (StandardScaler)
4. уменьшение размерности (PCA)
5. кластеризация (MiniBatchKMeans)

Предварительное преобразование призвано выравнить распределение признаков. Как было показано ранее, нулевые и отрицательные значения в данных ограничивают круг применимых нелинейных преобразований. Остаются Yeo-Johnson и QuantileTransformer с параметром output_distribution="normal". Также может быть применено линейное преобразование (RobustScaler). И последний, четвертый вариант — пропустить этот этап и сразу приступить к стандартизации.

Результат каждого преобразования анализируется методом Хорна для определения оптимального количества компонент PCA. После уменьшения размерности трансформированный набор данных передается в модель кластеризации MiniBatchKMeans. Количество кластеров выбирается в диапазоне от 2 до 8, результат оценивается по нескольким метрикам: индекс Калински-Харабаса, индекс Дэвиса-Болдина и индекс стабильности. Также вычисляется статистика Хопкинса как показатель целесообразности проведения кластеризации.

In [3]:
# Варианты предварительного преобразования
DISTORSION = {
    'None': FunctionTransformer(),  # Ничего не делает (трансформер-заглушка)
    'RobustScaler': RobustScaler(),
    'Yeo-Johnson': PowerTransformer(method="yeo-johnson"),
    'QuantileTransform': QuantileTransformer(output_distribution="normal", random_state=42)
}

# Варианты количества кластеров
N_CLUSTERS = range(2, 9)

## 2.1. Параллельный анализ Хорна на основе перестановок

Предназначен для поиска минимально необходимого числа компонент в методах уменьшения размерности PCA/SVD. Идея метода в том, чтобы оставить только те компоненты, которые объясняют больше дисперсии, чем шум.  

1. Генерируется случайная матрица шума того же размера на основе перестановок: независимо перемешиваются значения внутри каждого столбца. Это сохраняет исходную скошенность и распределение каждого признака, но полностью разрушает корреляционные связи.
2. Для нее считается PCA и получаются «случайные» собственные значения $\lambda_{rand, i}$.
3. Остаются только те реальные компоненты, где $\lambda_{real, i} > \lambda_{rand, i}$ (или 95-й процентиль шума). Результатом становится объективная нижняя граница для числа компонент, чьи собственные значения выше случайного шума.

## 2.2. Множество Парето для поиска лучшего решения

Для оценки качества проведенной кластеризации используются две нетребовательные к вычислительным ресурсам метрики: индекс Калински-Харабаса ($CH$) и индекс Дэвиса-Болдина ($DB$).

Самый простой способ скомбинировать обе метрики — ранжировать значения двух индексов и использовать сумму рангов. Это так называемый метод Борда — хорошая базовая эвристика, так как она не зависит от разных масштабов и единиц измерения метрик. Однако она имеет несколько серьезных недостатков, которые в итоге выберут не самое качественное, а просто «удобное» решение.

1. **Вырожденные кластеры.**
   Метрики $CH$ и $DB$ могут давать аномально высокие оценки, если алгоритм выделил один гигантский кластер и несколько микро-кластеров из выбросов. Метрики будут отличными, но такая кластеризация не имеет практической пользы.
2. **Проклятие размерности.**
   Значения $CH$ и $DB$ зависят от размерности пространства, которое возвращает PCA. В пространствах меньшей размерности точки математически всегда лежат плотнее, что дает естественное (но ложное) преимущество в $DB$.
3. **Потеря дистанции при переходе к рангам.**
   Ранги учитывают только порядок, но не реальные значения. Разница между первым и вторым местами по метрике может быть ничтожной, а между вторым и третьим — 50\%. Ранги превращают эти перепады в одинаковые шаги, стирая информацию о явных лидерах.

Для выбора лучшего варианта по нескольким метрикам существует более продвинутая техника — поиск множества Парето, то есть таких конфигураций, которые невозможно улучшить по одной метрике без ухудшения другой.

* Конфигурация A доминирует над B, если $CH_A \ge CH_B$ и $DB_A \le DB_B$ (при этом хотя бы одно строго).
* Конфигурации, над которыми никто не доминирует, формируют границу Парето. Из них финальный вариант выбирается визуально.
* Техника легко масштабируется на любое число метрик.

Композитная метрика получается через MinMax нормализацию индексов, их значения нормируются в диапазон $[0, 1]$:

$$CH_{norm} = \frac{CH - \min(CH)}{\max(CH) - \min(CH)}, \quad DB_{norm} = \frac{\max(DB) - DB}{\max(DB) - \min(DB)}$$

$$\text{Score} = 0.5 \cdot CH_{norm} + 0.5 \cdot DB_{norm}$$

Такой подход сохраняет реальный отрыв качественных моделей от средних.


## 2.3. Классическое ML-решение с масштабированием столбцов

### efficientnet-b7

In [ ]:
%%time

# efficientnet-b7
# Подвыборка 30% для калибровки
efficientnet_components = vwc.get_optimal_pca_components(
    efficientnet_cleaned, 
    DISTORSION, 
    n_components_eval=512, 
    sample_frac=0.3, 
    n_jobs=4
)
print(efficientnet_components)
# {'None': 376, 'RobustScaler': 376, 'Yeo-Johnson': 375, 'QuantileTransform': 375} # sample_frac=1.0
# {'None': 372, 'RobustScaler': 372, 'Yeo-Johnson': 370, 'QuantileTransform': 371} # sample_frac=0.3

Трансформеры: 100%|██████████| 4/4 [00:00<00:00, 109.53it/s]


{'None': np.int64(372), 'RobustScaler': np.int64(372), 'Yeo-Johnson': np.int64(370), 'QuantileTransform': np.int64(371)}
CPU times: total: 13 s
Wall time: 34min 12s


In [ ]:
if 'efficientnet_components' not in locals():
    efficientnet_components = {'None': 376, 'RobustScaler': 376, 'Yeo-Johnson': 375, 'QuantileTransform': 375} # sample_frac=1.0

In [5]:
%%time

efficientnet_metrics = vwc.clustering_measure(
    efficientnet_cleaned, 
    DISTORSION, 
    efficientnet_components, 
    N_CLUSTERS,
    n_jobs=4,
    random_state=42
)

Трансформеры:   0%|          | 0/4 [00:00<?, ?it/s]

Трансформеры: 100%|██████████| 4/4 [1:35:38<00:00, 1434.52s/it]

CPU times: total: 9h 54min 23s
Wall time: 1h 35min 38s


In [6]:
# Множество Парето - конфигурации, которые невозможно улучшить по одной метрике без ухудшения другой.
# Конфигурация A доминирует над B, если метрики A не хуже метрик B, но хотя бы одна из метрик строго лучше.
# Конфигурации, над которыми никто не доминирует, формируют границу Парето. Из них финальный вариант 
# выбирается визуально.

efficientnet_pareto_leaders = vwc.pareto_decision(efficientnet_metrics)
print(f"Найдено {len(efficientnet_pareto_leaders)} Парето-оптимальных решений:")

efficientnet_pareto_leaders

Найдено 3 Парето-оптимальных решений:


,transformer,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
21,QuantileTransform,0.769127,371,0.715164,2,31966.403530,2.981808,0.933581,0.030575,0.223709,0.979453
14,Yeo-Johnson,0.775955,370,0.708703,2,32388.746870,2.995406,0.908920,0.039777,0.228834,0.965778
7,RobustScaler,0.771355,372,0.734888,2,31694.023762,2.829767,0.803794,0.070488,0.163766,0.898779


In [8]:
efficientnet_metrics.to_csv(os.path.join('temp', 'efficientnet_metrics.csv'), index=False)

**Итоги.** Лучшее решение предлагает всего 2 кластера. Статистика Хопкинса = 0.715, данные содержат выраженные группы (H>0.7). Стабильность кластеризации отличная (>0.85): mean=0.936, std=0.032.

### osnet

In [ ]:
%%time

# osnet
osnet_components = vwc.get_optimal_pca_components(
    osnet_cleaned, 
    DISTORSION, 
    n_components_eval=100,
    sample_frac=1.0, 
    n_jobs=4
)
print(osnet_components)
# {'None': 82, 'RobustScaler': 82, 'Yeo-Johnson': 75, 'QuantileTransform': 75}  # sample_frac=1.0

Трансформеры: 100%|██████████| 4/4 [00:00<00:00,  5.60it/s]


{'None': np.int64(82), 'RobustScaler': np.int64(82), 'Yeo-Johnson': np.int64(75), 'QuantileTransform': np.int64(75)}
CPU times: total: 7.98 s
Wall time: 13min 15s


In [9]:
if 'osnet_components' not in locals():
    osnet_components = {'None': 82, 'RobustScaler': 82, 'Yeo-Johnson': 75, 'QuantileTransform': 75}

In [10]:
%%time

osnet_metrics = vwc.clustering_measure(
    osnet_cleaned, 
    DISTORSION, 
    osnet_components, 
    N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Трансформеры: 100%|██████████| 4/4 [34:59<00:00, 524.84s/it]

CPU times: total: 2h 2min 55s
Wall time: 34min 59s


In [11]:
osnet_pareto_leaders = vwc.pareto_decision(osnet_metrics)
print(f"Найдено {len(osnet_pareto_leaders)} Парето-оптимальных решений:")

osnet_pareto_leaders

Найдено 5 Парето-оптимальных решений:


,transformer,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
14,Yeo-Johnson,0.777965,75,0.808223,2,52885.742223,2.748261,0.753409,0.374756,0.437144,0.827023
24,QuantileTransform,0.698957,75,0.804349,5,30876.104666,2.708208,0.756413,0.139597,0.150908,0.639954
8,RobustScaler,0.809520,82,0.869147,3,32883.520886,2.699293,0.718662,0.206148,0.197286,0.615280
15,Yeo-Johnson,0.777965,75,0.808223,3,40336.862350,2.518838,0.554290,0.216170,0.201218,0.569018
18,Yeo-Johnson,0.777965,75,0.808223,6,28010.103058,2.903340,0.795206,0.130695,0.103758,0.561000


In [12]:
osnet_metrics.to_csv(os.path.join('temp', 'osnet_metrics.csv'), index=False)

**Итоги.** Оптимальных решений несколько, они очень разные: число кластеров 2, 5, 3 или 6. Максимальная композитная метрика относится к 2 кластерам, хотя в этом случае только индекс Калински-Харабаса показывает наилучшее значение. Статистика Хопкинса = 0.808, данные в osnet содержат выраженные группы (H>0.7). Стабильность кластеризации хорошая (>0.70), но с очень высоким разбросом: mean=0.753, std=0.375. По показателю стабильности и индексу Дэвиса-Болдина оптимальный вариант с 6 кластерами, но в этом случае наблюдается низкий индекс Калински-Харабаса.

### vdc_color

In [ ]:
%%time

# vdc_color
vdc_color_components = vwc.get_optimal_pca_components(
    vdc_color_cleaned, 
    DISTORSION, 
    n_components_eval=72,
    sample_frac=1.0, 
    n_jobs=4
)
print(vdc_color_components)
# {'None': 16, 'RobustScaler': 16, 'Yeo-Johnson': 16, 'QuantileTransform': 16}  # sample_frac=1.0

Трансформеры: 100%|██████████| 4/4 [00:00<00:00,  4.26it/s]


{'None': np.int64(16), 'RobustScaler': np.int64(16), 'Yeo-Johnson': np.int64(16), 'QuantileTransform': np.int64(16)}
CPU times: total: 3.17 s
Wall time: 5min 19s


In [14]:
if 'vdc_color_components' not in locals():
    vdc_color_components = {'None': 16, 'RobustScaler': 16, 'Yeo-Johnson': 16, 'QuantileTransform': 16}

In [15]:
%%time

vdc_color_metrics = vwc.clustering_measure(
    vdc_color_cleaned, 
    DISTORSION, 
    vdc_color_components, 
    N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Трансформеры: 100%|██████████| 4/4 [27:42<00:00, 415.50s/it]

CPU times: total: 1h 39min 46s
Wall time: 27min 42s


In [16]:
vdc_color_pareto_leaders = vwc.pareto_decision(vdc_color_metrics)
print(f"Найдено {len(vdc_color_pareto_leaders)} Парето-оптимальных решений:")

vdc_color_pareto_leaders

Найдено 3 Парето-оптимальных решений:


,transformer,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
14,Yeo-Johnson,0.646674,16,0.868169,2,134630.376992,1.658485,0.956979,0.028523,0.497444,0.871599
9,RobustScaler,0.660379,16,0.881526,4,112936.462756,1.430183,0.776737,0.133203,0.055578,0.741107
15,Yeo-Johnson,0.646674,16,0.868169,3,111824.461717,1.382112,0.516921,0.120263,0.061338,0.562108


In [17]:
vdc_color_metrics.to_csv(os.path.join('temp', 'vdc_color_metrics.csv'), index=False)

**Итоги.** Оптимальных решений несколько, число кластеров 2, 4 или 3. Максимальная композитная метрика относится к 2 кластерам, когда показывает наилучшее значение индекс Калинска-Харабаса, а также рекордные показатели стабильности. Статистика Хопкинса = 0.868, данные в vdc_color содержат выраженные группы (H>0.7). Стабильность кластеризации отличная (>0.85): mean=0.957, std=0.029.

### vdc_type

In [ ]:
%%time

# vdc_type
vdc_type_components = vwc.get_optimal_pca_components(
    vdc_type_cleaned, 
    DISTORSION, 
    n_components_eval=100,
    sample_frac=1.0, 
    n_jobs=4
)
print(vdc_type_components)
# {'None': 27, 'RobustScaler': 27, 'Yeo-Johnson': 26, 'QuantileTransform': 31}  # sample_frac=1.0

Трансформеры: 100%|██████████| 4/4 [00:00<00:00,  4.36it/s]


{'None': np.int64(27), 'RobustScaler': np.int64(27), 'Yeo-Johnson': np.int64(26), 'QuantileTransform': np.int64(31)}
CPU times: total: 6.28 s
Wall time: 11min 45s


In [19]:
if 'vdc_color_components' not in locals():
    vdc_type_components = {'None': 27, 'RobustScaler': 27, 'Yeo-Johnson': 26, 'QuantileTransform': 31}

In [20]:
%%time

vdc_type_metrics = vwc.clustering_measure(
    vdc_type_cleaned, 
    DISTORSION, 
    vdc_type_components, 
    N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Трансформеры: 100%|██████████| 4/4 [32:02<00:00, 480.51s/it]

CPU times: total: 1h 48min 47s
Wall time: 32min 2s


In [21]:
vdc_type_pareto_leaders = vwc.pareto_decision(vdc_type_metrics)
print(f"Найдено {len(vdc_type_pareto_leaders)} Парето-оптимальных решений:")
vdc_type_pareto_leaders

Найдено 6 Парето-оптимальных решений:


,transformer,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
14,Yeo-Johnson,0.888699,26,0.831492,2,139713.230184,1.674263,0.945265,0.035144,0.494483,0.904818
21,QuantileTransform,0.762417,31,0.835553,2,123251.837633,1.783848,0.973512,0.010953,0.495854,0.798937
16,Yeo-Johnson,0.888699,26,0.831492,4,110879.338178,1.576297,0.838137,0.136624,0.166377,0.771288
9,RobustScaler,0.912292,27,0.942945,4,94861.002133,1.558619,0.757844,0.040653,0.114527,0.657174
2,None,0.912292,27,0.942900,4,94861.034948,1.558619,0.757842,0.040651,0.114527,0.657173
15,Yeo-Johnson,0.888699,26,0.831492,3,114496.979511,1.608335,0.616488,0.202234,0.202669,0.582768


In [22]:
vdc_type_metrics.to_csv(os.path.join('temp', 'vdc_type_metrics.csv'), index=False)

**Итоги.** Оптимальных решений несколько, число кластеров 2, 4 или 3. Максимальная композитная метрика относится к 2 кластерам, когда наблюдаются наилучшие значения индекса Калинска-Харабаса и показателя стабильности. Статистика Хопкинса = 0.831, данные в vdc_type содержат выраженные группы (H>0.7). Стабильность кластеризации отличная (>0.85): mean=0.945, std=0.035.

## 2.4. Решение с L2-нормализацией по строкам

В обычном табличном ML каждый признак — это независимая переменная со своей шкалой. Перед подачей в модель их приводят к сопоставимому масштабу. Для дескрипторов свёрточных нейронных сетей (EfficientNet и др.) геометрия пространства признаков подчиняется законам компьютерного зрения, а это данные совершенно другой природы. Координаты дескриптора — это не независимые признаки, а координаты точки в едином пространстве, где направления (углы) несут семантический смысл, а отдельные координаты по отдельности — нет. Семантика сообщает, какие именно визуальные паттерны и в каких пропорциях активировались. Длина (норма) вектора зависит от силы активаций, которая определяется не только техническими параметрами картинки (контрастность, яркость, насыщенность, резкость, масштаб), но и сложностью паттернов, степенью уверенности сети и т.д. Алгоритм KMeans выделяет выпуклые, сферические кластеры, но в таком пространстве нет изолированных «шаров». Это непрерывное облако смыслов с локальными сгустками. Кроме того, большинство алгоритмов кластеризации работают исключительно через евклидово расстояние, в то время как для нейросетевых эмбеддингов важнее косинусное сходство, которое игнорирует длину и учитывает только углы между векторами, т.е. чистый визуальный смысл. 

Табличные преобразования (Yeo-Johnson, QuantileTransformer, StandardScaler) искажают нейросетевой смысл картинки, т.к. выравнивают распределения каналов по массиву изображений, меняют относительные пропорции между координатами вектора, а значит, и угол вектора. Вместо них более подходящим вариантом является построчная L2-нормализация, которая работает с каждым изображением индивидуально, деля вектор на его собственную длину. Таким образом все точки проецируются на поверхность единичной гиперсферы и сохраняются нетронутыми соотношения между всеми каналами, выраженные через косинусное расстояние. В итоге KMeans и PCA работают строго по углам, т.е. семантике, и игнорируют технические параметры картинок.

Новая, альтернативная последовательность действий включает в себя следующие шаги:

1. L2-нормализация по строкам.
2. Уменьшение размерности PCA.
3. Повторная L2-нормализация по строкам.
4. Уменьшение размерности UMAP с параметром metric='cosine'.
5. Кластеризация MiniBatchKMeans или HDBSCAN.

После PCA размерность данных уменьшается. Так как часть координат отброшена, длины векторов снова перестают быть строго равными 1.0. Повторная L2-нормализация возвращает сжатые векторы на единичную сферу уже в новом, сжатом пространстве.

### efficientnet-b7

In [ ]:
%%time

# efficientnet-b7
# Подвыборка 30% для калибровки
efficientnet_components_l2 = vwc.get_optimal_pca_components_l2(
    efficientnet_cleaned, 
    n_components_eval=512, 
    sample_frac=0.3
)
print(f'{efficientnet_components_l2=}')
# efficientnet_components_l2=376  # sample_frac=1.0
# efficientnet_components_l2=373  # sample_frac=0.3

efficientnet_components_l2=np.int64(373)
CPU times: total: 56min 37s
Wall time: 7min 59s


In [ ]:
%%time

efficientnet_metrics_l2 = vwc.clustering_measure_l2(
    efficientnet_cleaned, 
    n_components=efficientnet_components_l2, 
    clusters=N_CLUSTERS,
    n_jobs=4,
    random_state=42
)

Перебор по сетке: 100%|██████████| 7/7 [00:00<00:00, 145.85it/s]


CPU times: total: 1h 11min 30s
Wall time: 13min 16s


In [ ]:
# Множество Парето - конфигурации, которые невозможно улучшить по одной метрике без ухудшения другой.
# Конфигурация A доминирует над B, если метрики A не хуже метрик B, но хотя бы одна из метрик строго лучше.
# Конфигурации, над которыми никто не доминирует, формируют границу Парето. Из них финальный вариант 
# выбирается визуально.

efficientnet_pareto_leaders_l2 = vwc.pareto_decision(efficientnet_metrics_l2)
print(f"Найдено {len(efficientnet_pareto_leaders_l2)} Парето-оптимальных решений:")

efficientnet_pareto_leaders_l2

Найдено 1 Парето-оптимальных решений:


,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
0,0.755942,373,0.701745,2,19371.11513,4.443277,0.941887,0.032104,0.365875,1.0


In [ ]:
efficientnet_metrics_l2.to_csv(os.path.join('temp', 'efficientnet_metrics_l2.csv'), index=False)

### osnet

In [ ]:
%%time

# osnet
osnet_components_l2 = vwc.get_optimal_pca_components_l2(
    osnet_cleaned, 
    n_components_eval=100,
    sample_frac=1.0
)
print(f'{osnet_components_l2=}')
# osnet_components_l2=30  # sample_frac=1.0

osnet_components_l2=np.int64(30)
CPU times: total: 30min 46s
Wall time: 5min 18s


In [ ]:
%%time

osnet_metrics_l2 = vwc.clustering_measure_l2(
    osnet_cleaned, 
    n_components=osnet_components_l2, 
    clusters=N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Перебор по сетке: 100%|██████████| 7/7 [00:00<00:00, 130.82it/s]


CPU times: total: 26min 26s
Wall time: 7min 13s


In [ ]:
osnet_pareto_leaders_l2 = vwc.pareto_decision(osnet_metrics_l2)
print(f"Найдено {len(osnet_pareto_leaders_l2)} Парето-оптимальных решений:")

osnet_pareto_leaders_l2

Найдено 1 Парето-оптимальных решений:


,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
0,0.764219,30,0.818251,2,87809.178608,2.144632,0.797386,0.374063,0.444208,1.0


In [ ]:
osnet_metrics_l2.to_csv(os.path.join('temp', 'osnet_metrics_l2.csv'), index=False)

### vdc_color

In [ ]:
%%time

# vdc_color
vdc_color_components_l2 = vwc.get_optimal_pca_components_l2(
    vdc_color_cleaned, 
    n_components_eval=72,
    sample_frac=1.0
)
print(f'{vdc_color_components_l2=}')
# vdc_color_components=5

vdc_color_components_l2=np.int64(5)
CPU times: total: 10min 45s
Wall time: 1min 51s


In [ ]:
%%time

vdc_color_metrics_l2 = vwc.clustering_measure_l2(
    vdc_color_cleaned, 
    n_components=vdc_color_components_l2, 
    clusters=N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Перебор по сетке: 100%|██████████| 7/7 [00:00<00:00, 6972.25it/s]


CPU times: total: 12.8 s
Wall time: 22.9 s


In [ ]:
vdc_color_pareto_leaders_l2 = vwc.pareto_decision(vdc_color_metrics_l2)
print(f"Найдено {len(vdc_color_pareto_leaders_l2)} Парето-оптимальных решений:")

vdc_color_pareto_leaders_l2

Найдено 1 Парето-оптимальных решений:


,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
0,0.552762,5,0.904963,2,524856.792226,0.829979,0.998361,0.000773,0.48687,1.0


In [ ]:
vdc_color_metrics_l2.to_csv(os.path.join('temp', 'vdc_color_metrics_l2.csv'), index=False)

### vdc_type

In [ ]:
%%time

# vdc_type
vdc_type_components_l2 = vwc.get_optimal_pca_components_l2(
    vdc_type_cleaned, 
    n_components_eval=100,
    sample_frac=1.0
)
print(f'{vdc_type_components_l2=}')
# 

vdc_type_components_l2=np.int64(17)
CPU times: total: 28min 49s
Wall time: 4min 25s


In [ ]:
%%time

vdc_type_metrics_l2 = vwc.clustering_measure_l2(
    vdc_type_cleaned, 
    n_components=vdc_type_components_l2, 
    clusters=N_CLUSTERS,
    n_jobs=8,
    random_state=42
)

Перебор по сетке: 100%|██████████| 7/7 [00:00<00:00, 125.35it/s]


CPU times: total: 25min 6s
Wall time: 6min 53s


In [ ]:
vdc_type_pareto_leaders_l2 = vwc.pareto_decision(vdc_type_metrics_l2)
print(f"Найдено {len(vdc_type_pareto_leaders_l2)} Парето-оптимальных решений:")

vdc_type_pareto_leaders_l2

Найдено 3 Парето-оптимальных решений:


,pca_variance,n_pca_components_actual,hopkins_stat,n_clusters,calinski_harabasz,davies_bouldin,stability,stability_std,min_cluster_pct,composite_score
0,0.857222,17,0.831655,2,137140.502322,1.712829,0.993890,0.002109,0.442255,0.700000
2,0.857222,17,0.831655,4,116620.003127,1.507029,0.852397,0.169672,0.181812,0.674611
3,0.857222,17,0.831655,5,103803.476163,1.486192,0.724255,0.143599,0.100544,0.462359


In [ ]:
vdc_type_metrics_l2.to_csv(os.path.join('temp', 'vdc_type_metrics_l2.csv'), index=False)